# Markov Chains

---
In class today we will be implementing a Markov chain to process sentences

---
## Learning Objectives

1. Students will be able to explain the Markov Chain process
1. Implement a Markov Chain


Markov Chains represent a series of events following the Markov Property: future states are memory-less in that they depend only on the current state. This can be expanded to the idea of variable order Markov models where there is a variable-length memory (eg. 1st order Markov Model). Markov models consist of fully observable states. 

> A common example of this is in predicting the weather: We can clearly see the current weather and would like to predict tomorrow's weather. This is also applicable to biology with one case being CpG islands. 

Our goal today will be to implement a Markov model built from words. For our example text, we will use the classic example of Dr. Seuss because of the repetitive nature of the text.

---
## Train Markov model

For our initial implementation of the Markov Model, we will use the simple example of Dr. Seuss: "One fish two fish red fish blue fish."



In [11]:
def build_markov_model(markov_model, new_text):
    '''
    Function to build or add to a 1st order Markov model given a string of text
    We will store the markov model as a dictionary of dictionaries
    The key in the outer dictionary represents the current state
    and the inner dictionary represents the next state with their contents containing
    the transition probabilities.
    Note: This would be easier to read if we were to build a class representation
           of the model rather than a dictionary of dictionaries, but for simplicitiy
           our implementation will just use this structure.

    Args:
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)
        new_text (str): a string to build or add to the moarkov_model

    Returns:
        markov_model (dict of dicts): an updated markov_model

    Pseudocode:
        Add artificial states for start and end
        For each word in text:
            Increment markov_model[word][next_word]

    '''

    # establish artificial start and end states
    end = '*E*'
    start = ['*S']

    # split the text by any whitespace character
    text = new_text.split()

    full_list = start + text
    # Loop over each word in the text and enumerate to keep track of positions
    for i, word in enumerate(full_list):
        # if the word is in the start, update the inner dict with value
        if word == start:
            if start in markov_model:
                if text[i] in markov_model[start]:
                    markov_model[start][text[i]] += 1
                else:
                    markov_model[start][text[i]] = 1
            else:
                markov_model[start] = {text[i]: 1}
        # if the word is at the end, update the inner dict with the value and exit the loop
        elif i == len(full_list) - 1:
            # if the word is already in the model, update it without overwriting the current values
            if word in markov_model:
                markov_model[word][end] = 1
            # if it isn't, add it
            else:
                markov_model[word] = {end: 1}
            break

        # if the word is in the dictionary, update the inner value
        elif word in markov_model:
            # if the following word is already in the inner dict, add one to it's value
            if text[i] in markov_model[word]:
                markov_model[word][text[i]] += 1
            # if the following word isn't already in the inner dict, initialize the value
            else:
                markov_model[word][text[i]] = 1
        # if this is the first time we've encountered the word, initialize it (the value of the inner dict will always be initialized to one because this is the first time we've encountered the word. That means it will always be the first time the next word occurs after that word.
        else:
            markov_model[word] = {text[i]: 1}


    return markov_model

In [12]:
markov_model = dict()
text = "one fish two fish red fish blue fish"
markov_model = build_markov_model(markov_model, text)
print (markov_model)

{'*S': {'one': 1}, 'one': {'fish': 1}, 'fish': {'two': 1, 'red': 1, 'blue': 1, '*E*': 1}, 'two': {'fish': 1}, 'red': {'fish': 1}, 'blue': {'fish': 1}}


###  Nth order Markov chain
In the above model, each event or word is output from only the previous state with no memory of any prior states. While this is useful in some cases, typical biological applications of Markov chains require higher-order models to accurately capture what we know about a system. For instance, in attempting to identify coding regions of a genome, we know that open reading frames (ORFs) contain codon triplets, and so a third or sixth order Markov chain would better describe these regions. Here you will implement a generalized form of our previous Markov Chain to allow for Nth order chains.


In [13]:
def build_markov_model(markov_model, new_text, order=1):
    '''
    Function to build or add to a Nth order Markov model given a string of text

    Args:
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)
            or None if a new model is being built
        new_text (str): a string to build or add to the markov_model
        order (int): the number of previous states to consider for the model

    Returns:
        markov_model (dict of dicts): an updated/new markov_model
    '''

    # establish artificial start and end states
    start = '*S*'
    end = '*E*'

    # split the text by any whitespace character
    text = new_text.split()
    start_list = ['*S*'] * order
    # add this list to the text
    full_list = start_list + text

    # create a start list to account for the nth order, this ensures that the index of the order_list is always one behind the text list
    if order > 1:

    # initialize all nth order value pairs list
        order_list = []

    # find all possible n order length combinations of words
    # this index only goes until the second to last
        for i in range(len(full_list) - (order - 1)):
            order_value = tuple(full_list[i: i + order])
            order_list.append(order_value)


        for i, order_words in enumerate(order_list):
            if order_words == ('*S*', ) * order:
                if order_words in markov_model:
                    if text[i] in markov_model[order_words]:
                        markov_model[order_words][text[i]] += 1
                    else:
                        markov_model[order_words][text[i]] = 1
                else:
                    markov_model[order_words] = {text[i]: 1}

            elif i == len(order_list) - 1:
                if order_words in markov_model:
                    markov_model[order_words][end] = 1

                else:
                    markov_model[order_words] = {end: 1}
                break

            elif order_words in markov_model:

                if text[i] in markov_model[order_words]:
                    markov_model[order_words][text[i]] += 1
                else:
                    markov_model[order_words][text[i]] = 1

            else:
                markov_model[order_words] = {text[i]: 1}


    else:
    # for an order of 1
    # Loop over each word in the text and enumerate to keep track of positions
        for i, word in enumerate(full_list):
        # if the word is in the start, update the inner dict with value
            if word == start:
                if start in markov_model:
                    if text[i] in markov_model[start]:
                        markov_model[start][text[i]] += 1
                    else:
                        markov_model[start][text[i]] = 1
                else:
                    markov_model[start] = {text[i]: 1}
            # if the word is at the end, update the inner dict with the value and exit the loop
            elif i == len(full_list) - 1:
            # if the word is already in the model, update it without overwriting the current values
                if word in markov_model:
                    markov_model[word][end] = 1
            # if it isn't, add it
                else:
                    markov_model[word] = {end: 1}
                break

        # if the word is in the dictionary, update the inner value
            elif word in markov_model:
            # if the following word is already in the inner dict, add one to it's value
                if text[i] in markov_model[word]:
                    markov_model[word][text[i]] += 1
            # if the following word isn't already in the inner dict, initialize the value
                else:
                    markov_model[word][text[i]] = 1
        # if this is the first time we've encountered the word, initialize it (the value of the inner dict will always be initialized to one because this is the first time we've encountered the word. That means it will always be the first time the next word occurs after that word.
            else:
                markov_model[word] = {text[i]: 1}

    return markov_model

In [14]:
markov_model = dict()
text = "one fish two fish red fish blue fish"
markov_model = build_markov_model(markov_model, text, order=2)
markov_model

{('*S*', '*S*'): {'one': 1},
 ('*S*', 'one'): {'fish': 1},
 ('one', 'fish'): {'two': 1},
 ('fish', 'two'): {'fish': 1},
 ('two', 'fish'): {'red': 1},
 ('fish', 'red'): {'fish': 1},
 ('red', 'fish'): {'blue': 1},
 ('fish', 'blue'): {'fish': 1},
 ('blue', 'fish'): {'*E*': 1}}

## Generate text from Markov Model

Markov models are "generative models". That is, the probability states in the model can be used to generate output following the conditional probabilities in the model.

We will now generate a sequence of text from the Markov model. For this section, I recommend using np.random.choice, which allows for you to provide a probability distribution for drawing the next edge in the chain.

In [15]:
import numpy as np

def get_next_word(current_word, markov_model):
    '''
    Function to randomly move a valid next state given a markov model
    and a current state (word)

    Args:
        current_word (tuple) (string): a word that exists in our model
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)

    Returns:
        next_word (str): a randomly selected next word based on transition probabilies

    Pseudocode:
        Calculate transition probilities for all next states from a given state (counts/sum)
        Randomly draw from these to generate the next state

    '''

    for orders, next_words in markov_model.items():
        total_obs = 0
        for i, next_word in enumerate(next_words):
            total_obs += next_words[next_word]
            if i == len(next_words) - 1:
                for next_word in next_words:
                    next_words[next_word] = next_words[next_word] / total_obs

    next_words = list(markov_model[current_word].keys())
    probabilities = list(markov_model[current_word].values())
    word_choice = np.random.choice(next_words, p=probabilities)
    return word_choice


def generate_random_text(markov_model, seed):
    '''
    Function to generate text given a markov model

    Args:
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)

    Returns:
        sentence (str): a randomly generated sequence given the model

    Pseudocode:
        Initialize sentence at start state
        Until End State:
            append get_next_word(current_word, markov_model)
        Return sentence

    '''
    np.random.seed(seed)
    markov_keys = list(markov_model.keys())
    current_word = markov_keys[0]
    generated_text = []

    if type(current_word) != str:
        while True:
            next_word = get_next_word(current_word, markov_model)
            if next_word == '*E*':
                break

            generated_text.append(next_word)
            current_word = current_word[1:] + (next_word,)

    else:
        while True:
            next_word = get_next_word(current_word, markov_model)
            if next_word == '*E*':
                break

            generated_text.append(next_word)
            current_word = next_word


    sentence = " ".join(generated_text)
    return sentence

---

## All the Fish
Up till now, you have only been working with a line or two of the Dr. Seuss' _One Fish, Two Fish_. Now, I want you to build a model using the whole book and try different orders of Markov models.

In [16]:
full_onefish_model = dict()
file = open("one_fish_two_fish.txt", mode='r', encoding = 'utf-8')
full_text = ""
for line in file:
    s_line = line.rstrip()
    full_text = full_text + ' ' + s_line

full_onefish_model = build_markov_model(full_onefish_model, full_text, order=1)

print (generate_random_text(markov_model,seed=7))

one fish two fish red fish blue fish


---
## Shakespeare

Now, let's play around with some Shakespeare.

In [17]:
sonnet_markov_model = dict()
file = open('sonnets.txt', mode='r', encoding='utf-8')
sonnet = ""
count = 0
for line in file:
    s_line = line.rstrip()
    if s_line == "":
        sonnet_markov_model = build_markov_model(sonnet_markov_model, sonnet, order = 3)
        sonnet = ""
    else:
        sonnet = sonnet + ' ' + line
 
print(generate_random_text(sonnet_markov_model,seed=7))

When most I wink, then do mine eyes best see, For all the day they view things unrespected; But when I sleep, in dreams they look on thee, And darkly bright, are bright in dark directed. Then thou, whose shadow shadows doth make bright, How would thy shadow's form form happy show To the clear day with thy much clearer light, When to unseeing eyes thy shade shines so! How would, I say, mine eye saith true, And that your love taught it this alchemy, To make of monsters and things indigest Such cherubins as your sweet self resemble, Creating every bad a perfect best, As fast as thou shalt wane, so fast thou grow'st, In one of thine, from that which thou hast done: Roses have thorns, and silver fountains mud: Clouds and eclipses stain both moon and sun, And loathsome canker lives in sweetest bud. All men make faults, and even I in this, Authorizing thy trespass with compare, Myself corrupting, salving thy amiss, Excusing thy sins more than thy sins are; For to thy sensual fault I bring in 